In [1]:
import torch
import torch.nn as nn

def compute_rope_frequencies(dim, max_seq_len, theta=10000.0):
    """
    Precompute the frequencies for the rotary embeddings.
    """
    # dim must be even
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Generate the inverse frequencies: theta^(-2i/d)
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float().to(device) / dim))
    
    # Generate the positions (m)
    t = torch.arange(max_seq_len, device=device).type_as(inv_freq)
    
    # Outer product to get the angles (m * theta)
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    
    # Convert to polar form (cos, sin)
    # We repeat each frequency to match the dimension of the embedding
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos(), emb.sin()

def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(x, cos, sin):
    """
    Applies RoPE to the input tensor x.
    x: [batch, heads, seq_len, head_dim]
    cos, sin: [seq_len, head_dim]
    """
    # Align shapes for broadcasting
    cos = cos[:x.shape[2], :].unsqueeze(0).unsqueeze(1)
    sin = sin[:x.shape[2], :].unsqueeze(0).unsqueeze(1)
    
    # Apply the rotation: x * cos(m*theta) + rotate_half(x) * sin(m*theta)
    return (x * cos) + (rotate_half(x) * sin)

# --- Quick Test ---
batch, heads, seq_len, head_dim = 1, 8, 128, 64
q = torch.randn(batch, heads, seq_len, head_dim)

cos, sin = compute_rope_frequencies(head_dim, seq_len)
q_rotated = apply_rope(q, cos, sin)

print(f"Original shape: {q.shape}")
print(f"Rotated shape: {q_rotated.shape}")

Original shape: torch.Size([1, 8, 128, 64])
Rotated shape: torch.Size([1, 8, 128, 64])
